In [69]:
import random
from typing import List, Tuple
from copy import deepcopy


class Tablerowumpus:
    
    def __init__(self, matrix=None, rows=6, cols=6):
        if matrix is not None:
            self.setMatrix(matrix)
            self.rows = len(matrix)
            self.cols = len(matrix[0]) if matrix else 0
        else:
            self.rows = rows
            self.cols = cols
            self.matrix = [[0 for _ in range(cols)] for _ in range(rows)]
        
        self.wumpus_pos = None
        self.oro_pos = None
        self.agent_pos = (self.rows - 1, 0)  # Agente siempre empieza en la esquina inferior izquierda
        self.hole_positions = []
        self.visited_positions = set()  # Para rastrear las posiciones visitadas

    def __eq__(self, other) -> bool:
        if not isinstance(other, Tablerowumpus):
            return False
        return self.matrix == other.matrix
    
    
    def setMatrix(self, matrix):
        self.matrix = deepcopy(matrix)
    
    
    def getMatrix(self) -> List[List]:
        return deepcopy(self.matrix)
    
    
    def placeTile(self, row: int, col: int, tile: int):
        current = self.matrix[row][col]
        if current == 0:
            self.matrix[row][col] = tile
        else:
            self.matrix[row][col] = self.combine_effects(current, tile)
            
    def setup_board(self):
        # Colocar el agente en la esquina inferior izquierda
        self.agent_pos = (self.rows - 1, 0)
        self.placeTile(self.agent_pos[0], self.agent_pos[1], 1)  # 1 AGENTE
        
       # Colocar Wumpus en una posición aleatoria que no sea la esquina inferior izquierda
        while True:
            self.wumpus_pos = self.get_random_position()
            if self.wumpus_pos != self.agent_pos:
                break
        self.placeTile(self.wumpus_pos[0], self.wumpus_pos[1], 2)  # 2 WUMPUS
        
        # Colocar oro (con restricción)
        while True:
            self.oro_pos = self.get_random_position()
            if not self.is_adjacent(self.oro_pos, self.agent_pos) and self.oro_pos != self.wumpus_pos:
                break
        self.placeTile(self.oro_pos[0], self.oro_pos[1], 4)  # 4 ORO
        
        # Colocar dos hoyos en posiciones aleatorias que no sean adyacentes al Wumpus o al agente
        for _ in range(2):
            while True:
                hole_pos = self.get_random_position()
                if not (self.is_adjacent(hole_pos, self.wumpus_pos) or hole_pos == self.agent_pos):
                    self.placeTile(hole_pos[0], hole_pos[1], 6)  # 6 HOYO
                    self.hole_positions.append(hole_pos)
                    break
        
        # Generar vecinos (hedor y brisa)
        self.generate_neighbors()

    def placeTile(self, row: int, col: int, tile: int):
        current = self.matrix[row][col]
        if current == 0:
            self.matrix[row][col] = tile
        else:
            self.matrix[row][col] = self.combine_effects(current, tile)
        
    def combine_effects(self, effect1: int, effect2: int) -> int:
        effects = {effect1, effect2}
        if 1 in effects:  # Si uno de los efectos es el agente
            return 1  # Prioriza mostrar al agente
        if 3 in effects and 5 in effects and 4 in effects:
            return 10  # Hedor (3), Brisa (5) y Oro (4)
        elif 3 in effects and 4 in effects:
            return 9   # Hedor (3) y Oro (4)
        elif 5 in effects and 4 in effects:
            return 8   # Brisa (5) y Oro (4)
        elif 3 in effects and 5 in effects:
            return 7   # Hedor (3) y Brisa (5)
        else:
            return max(effect1, effect2)
        
    def get_random_position(self):
        while True:
            row = random.randint(0, self.rows - 1)
            col = random.randint(0, self.cols - 1)
            if self.matrix[row][col] == 0 and (row, col) != self.agent_pos:
                return (row, col)
            
    def is_adjacent(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1]) == 1

    def generate_neighbors(self):
        # Generar hedor alrededor del Wumpus
        self.generate_neighbor_effect(self.wumpus_pos, 3)  # 3 HEDOR

        # Generar brisa alrededor de los hoyos
        for hole in self.hole_positions:
            self.generate_neighbor_effect(hole, 5)  # 5 BRISA

    def generate_neighbor_effect(self, pos, effect):
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in directions:
            nx, ny = pos[0] + dx, pos[1] + dy
            if 0 <= nx < self.rows and 0 <= ny < self.cols:
                self.placeTile(nx, ny, effect)

    def move_agent(self, direction):
        row, col = self.agent_pos
        
        if direction == 'up' and self.canMoveUp(row, col):
            new_pos = (row - 1, col)
        elif direction == 'down' and self.canMoveDown(row, col):
            new_pos = (row + 1, col)
        elif direction == 'left' and self.canMoveLeft(row, col):
            new_pos = (row, col - 1)
        elif direction == 'right' and self.canMoveRight(row, col):
            new_pos = (row, col + 1)
        else:
            return False
        
        # Verificar si el nuevo movimiento es hacia una celda visitada
        if new_pos in self.visited_positions and not self.is_necessary_to_return(new_pos):
            return False
        
        # Guardar el contenido de la celda actual
        current_cell_content = self.matrix[row][col]

         # Mover al agente
        self.matrix[new_pos[0]][new_pos[1]] = self.combine_effects(1, self.matrix[new_pos[0]][new_pos[1]])
        self.agent_pos = new_pos

        # Restaurar el contenido de la celda anterior
        if current_cell_content != 1:  # Si no era solo el agente
            self.matrix[row][col] = current_cell_content
        else:
            # Si era solo el agente, restauramos los efectos originales
            original_effects = self.get_original_effects(row, col)
            self.matrix[row][col] = original_effects

        return True

    def get_original_effects(self, row, col):
        effects = 0
        if self.is_adjacent((row, col), self.wumpus_pos):
            effects |= 3  # Añadir hedor (3)
        for hole in self.hole_positions:
            if self.is_adjacent((row, col), hole):
                effects |= 5  # Añadir brisa (5)
        if (row, col) == self.oro_pos:
            effects |= 4  # Añadir oro (4)
        return effects if effects != 0 else 0

    def is_necessary_to_return(self, position):
        agent_row, agent_col = self.agent_pos
        
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                nx, ny = agent_row + dx, agent_col + dy
                if (nx, ny) == position: 
                    continue
                
                if self.matrix[nx][ny] == 6:  
                    return True
        
        return False
    
    def move_random_hole(self):
        if not self.hole_positions:
            return  # No hay hoyos para mover

        # Elegir un hoyo aleatorio para mover
        hole_to_move = random.choice(self.hole_positions)
        self.hole_positions.remove(hole_to_move)

        # Eliminar el hoyo y sus brisas
        self.matrix[hole_to_move[0]][hole_to_move[1]] = 0
        self.remove_neighbor_effect(hole_to_move, 5)  # 5 es brisa

        # Encontrar una nueva posición vacía
        new_pos = self.get_random_position()
        self.hole_positions.append(new_pos)

        # Colocar el nuevo hoyo y sus brisas
        self.placeTile(new_pos[0], new_pos[1], 6)  # 6 es hoyo
        self.generate_neighbor_effect(new_pos, 5)  # 5 es brisa

    def remove_neighbor_effect(self, pos, effect):
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in directions:
            nx, ny = pos[0] + dx, pos[1] + dy
            if 0 <= nx < self.rows and 0 <= ny < self.cols:
                current_cell = self.matrix[nx][ny]
                if current_cell == effect:
                    # Verificar si hay otro agujero cerca que justifique mantener la brisa
                    if not self.should_keep_effect(nx, ny, effect):
                        self.matrix[nx][ny] = 0
                elif current_cell in [7, 8, 9, 10]:  # Combinaciones
                    new_cell = self.remove_effect(current_cell, effect)
                    self.matrix[nx][ny] = new_cell

    def should_keep_effect(self, x, y, effect):
        if effect != 5:  # Solo nos preocupamos por las brisas (efecto 5)
            return False
        
        # Verificar si hay algún agujero adyacente que justifique mantener la brisa
        for hole in self.hole_positions:
            if self.is_adjacent((x, y), hole):
                return True
        return False

    def remove_effect(self, cell, effect):
        if cell == 7 and effect == 3:  # Hedor y Brisa -> Brisa
            return 5
        elif cell == 7 and effect == 5:  # Hedor y Brisa -> Hedor
            return 3
        elif cell == 8 and effect == 5:  # Brisa y Oro -> Oro
            return 4
        elif cell == 9 and effect == 3:  # Hedor y Oro -> Oro
            return 4
        elif cell == 10 and effect == 3:  # Hedor, Brisa y Oro -> Brisa y Oro
            return 8
        elif cell == 10 and effect == 5:  # Hedor, Brisa y Oro -> Hedor y Oro
            return 9
        else:
            return cell  # Si no hay cambio, devolver la celda original
                    
    def find_agent(self) -> Tuple[int, int]:
        for i in range(self.rows):
            for j in range(self.cols):
                if self.matrix[i][j] == 1:  # 1 es el agente
                    return (i, j)
        return None  # No se encontró al agente

    def canMoveUp(self, row: int, col: int) -> bool:
        return row > 0 and self.matrix[row-1][col] != 6  # 6 es un agujero

    def canMoveDown(self, row: int, col: int) -> bool:
        return row < self.rows - 1 and self.matrix[row+1][col] != 6

    def canMoveLeft(self, row: int, col: int) -> bool:
        return col > 0 and self.matrix[row][col-1] != 6

    def canMoveRight(self, row: int, col: int) -> bool:
        return col < self.cols - 1 and self.matrix[row][col+1] != 6
    
    def getAvailableMovesForMax(self, row: int, col: int) -> List[str]:
        moves = []
        if row > 0 and self.matrix[row-1][col] not in [2, 6]:  # No Wumpus, no hoyo
            moves.append('up')
        if row < self.rows - 1 and self.matrix[row+1][col] not in [2, 6]:
            moves.append('down')
        if col > 0 and self.matrix[row][col-1] not in [2, 6]:
            moves.append('left')
        if col < self.cols - 1 and self.matrix[row][col+1] not in [2, 6]:
            moves.append('right')
        return moves

    def getAvailableMovesForMin(self) -> List[Tuple[int, int]]:
        """
        Determina los movimientos disponibles para el entorno (jugador MIN).
        En este caso, el entorno puede 'mover' un hoyo a cualquier celda vacía.
        """
        moves = []
        for row in range(self.rows):
            for col in range(self.cols):
                if self.matrix[row][col] == 0:  # Celda vacía
                    moves.append((row, col))
        return moves

    def moveCanBeMade(self, player: int) -> bool:
        if player == 1:  # Agente
            return len(self.getAvailableMovesForMax(*self.agent_pos)) > 0
        else:  # Entorno (Wumpus)
            return len(self.getAvailableMovesForMin()) > 0

    def isGameOver(self) -> bool:
        agent_row, agent_col = self.agent_pos
        current_cell = self.matrix[agent_row][agent_col]

        if current_cell in [4, 8, 9, 10]:  # Oro
            print("¡El agente ha encontrado el oro! ¡Victoria!")
            return True
        elif current_cell == 2:  # Wumpus
            print("¡El Wumpus ha atrapado al agente! Derrota.")
            return True
        elif current_cell == 6:  # Hoyo
            print("¡El agente ha caído en un hoyo! Derrota.")
            return True

        return False
    
    def move(self, direction: str) -> bool:
        row, col = self.agent_pos
        new_row, new_col = row, col

        # Verifica si el movimiento es válido
        if direction == 'up' and row > 0:
            new_row = row - 1
        elif direction == 'down' and row < self.rows - 1:
            new_row = row + 1
        elif direction == 'left' and col > 0:
            new_col = col - 1
        elif direction == 'right' and col < self.cols - 1:
            new_col = col + 1
        else:
            return False  # Movimiento inválido

        # Guardar el contenido de la celda actual
        current_cell_content = self.matrix[row][col]

        # Mover al agente
        new_cell_content = self.matrix[new_row][new_col]
        self.matrix[new_row][new_col] = self.combine_effects(1, new_cell_content)
        self.agent_pos = (new_row, new_col)

        # Restaurar el contenido de la celda anterior
        if current_cell_content != 1:  # Si no era solo el agente
            self.matrix[row][col] = current_cell_content
        else:
            # Si era solo el agente, restauramos los efectos originales
            original_effects = self.get_original_effects(row, col)
            self.matrix[row][col] = original_effects

        # Verificar si el agente encuentra el oro o cae en un peligro
        if new_cell_content in [4, 8, 9, 10]:  # Oro o combinaciones con oro
            print("¡El agente ha encontrado el oro! ¡Victoria!")
            return True
        elif new_cell_content in [2, 6]:  # Wumpus o hoyo
            print("¡El agente ha caído en un peligro! Fin del juego.")
            return True

        return False  # El juego continúa

    def utility(self) -> float:
        agent_row, agent_col = self.agent_pos
        
        # Encontrar oro
        gold_pos = None
        for i in range(self.rows):
            for j in range(self.cols):
                if self.matrix[i][j] == [4, 8, 9, 10]:  # 4 es oro
                    gold_pos = (i, j)
                    break
            if gold_pos:
                break
        
        if not gold_pos:
            return -1000  # Penalización alta si no hay oro
        
        # Distancia Manhattan al oro
        distance_to_gold = abs(agent_row - gold_pos[0]) + abs(agent_col - gold_pos[1])
        
        # Recompensa por acercarse al oro
        gold_reward = 500 - (5 * distance_to_gold)
        
        # Penalización por peligro (Wumpus, hoyos, hedores y brisas)
        danger_penalty = 0
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                nx, ny = agent_row + dx, agent_col + dy
                if 0 <= nx < self.rows and 0 <= ny < self.cols:
                    if self.matrix[nx][ny] == 2:  # Wumpus
                        danger_penalty += 50
                    elif self.matrix[nx][ny] == 6:  # Hoyo
                        danger_penalty += 40
                    elif self.matrix[nx][ny] in [3, 5, 7]:  # Hedor o brisa
                        danger_penalty += 10
        
        return gold_reward - danger_penalty
        
    def get_best_move(self, moves):
        best_move = None
        best_utility = float('-inf')
        for move in moves:
            temp_state = self.copy()
            temp_state.move(move)
            move_utility = temp_state.utility()
            
            # Verificar si el movimiento nos acerca al oro
            if self.is_move_towards_gold(move):
                move_utility += 100  # Bonificación por moverse hacia el oro
            
            if move_utility > best_utility:
                best_utility = move_utility
                best_move = move
        return best_move

    def is_move_towards_gold(self, move):
        agent_row, agent_col = self.agent_pos
        gold_row, gold_col = self.oro_pos
        
        if move == 'up' and agent_row > gold_row:
            return True
        elif move == 'down' and agent_row < gold_row:
            return True
        elif move == 'left' and agent_col > gold_col:
            return True
        elif move == 'right' and agent_col < gold_col:
            return True
        return False

    def is_gold_nearby(self):
        agent_row, agent_col = self.agent_pos
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                nx, ny = agent_row + dx, agent_col + dy
                if 0 <= nx < self.rows and 0 <= ny < self.cols:
                    if self.matrix[nx][ny] in [4, 8, 9, 10]:  # Oro o combinaciones con oro
                        return True
        return False
    
    def copy(self):
        new_state = Tablerowumpus(matrix=self.getMatrix())  # Asegúrate de pasar la matriz aquí
        new_state.wumpus_pos = self.wumpus_pos
        new_state.oro_pos = self.oro_pos
        new_state.agent_pos = self.agent_pos
        new_state.hole_positions = self.hole_positions.copy()

        return new_state
    
    def get_html(self):
        html_string = "<style> img.game {width: 37px !important; height: 22px !important;}</style><table>"
        for row in self.matrix:
            html_string += "<tr>"
            for cell in row:
                content = self.get_content(cell)
                drawing = self.element_image[content]
                html_string += f'<td><img class="game" src="{drawing}" alt=""></img></td>'
            html_string += "</tr>"
        html_string += "</table>"
        return html_string

    def get_content(self, cell):
        content_map = {
            0: "CasillaVacia",
            1: "CasillaAgente",
            2: "CasillaWumpus",
            3: "CasillaHedor",
            4: "CasillaOro",
            5: "CasillaBrisa",
            6: "CasillaHueco",
            7: "CasillaBrisaHedor",
            8: "CasillaBrisaOro",
            9: "CasillaHedorOro",
            10: "CasillaBrisaHedorOro"
        }
        return content_map.get(cell, "CasillaVacia")

    element_image = {
        "CasillaVacia": "./ImagenesCasillasWumpus/CasillaVacia.png",
        "CasillaAgente": "./ImagenesCasillasWumpus/CasillaAgente.jpg",
        "CasillaWumpus": "./ImagenesCasillasWumpus/CasillaWumpus.jpg",
        "CasillaHedor": "./ImagenesCasillasWumpus/CasillaHedor.jpg",
        "CasillaOro": "./ImagenesCasillasWumpus/CasillaOro.jpg",
        "CasillaBrisa": "./ImagenesCasillasWumpus/CasillaBrisa.jpg",
        "CasillaHueco": "./ImagenesCasillasWumpus/CasillaHueco.jpg",
        "CasillaBrisaHedor": "./ImagenesCasillasWumpus/CasillaBrisaHedor.jpg",
        "CasillaBrisaOro": "./ImagenesCasillasWumpus/CasillaOroBrisa.jpg",
        "CasillaHedorOro": "./ImagenesCasillasWumpus/CasillaOroHedor.jpg",
        "CasillaBrisaHedorOro": "./ImagenesCasillasWumpus/CasillaOroHedorBrisa.jpg"
    }

In [70]:
def miniMax(state: Tablerowumpus, currentLevel: int, maxLevel: int, player: int, alpha: float, beta: float):
    if state.isGameOver() or currentLevel == maxLevel:
        return (state, state.utility(), None)
    
    if player == 1:  # Jugador MAX (Agente)
        best_value = float('-inf')
        best_move = None
        
        moves = state.getAvailableMovesForMax(*state.agent_pos)
        
        for move in moves:
            new_state = state.copy()
            new_state.move(move)
            _, value, _ = miniMax(new_state, currentLevel + 1, maxLevel, 2, alpha, beta)
            
            if value > best_value:
                best_value = value
                best_move = move
            
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break
        
        return (state, best_value, best_move)

    else:  # Jugador MIN (Entorno)
        best_value = float('inf')
        
        for _ in range(min(3, len(state.getAvailableMovesForMin()))):
            new_state = state.copy()
            new_state.move_random_hole()
            _, value, _ = miniMax(new_state, currentLevel + 1, maxLevel, 1, alpha, beta)
            
            if value < best_value:
                best_value = value
            
            beta = min(beta, best_value)
            if beta <= alpha:
                break
        
        return (state, best_value, None)

In [71]:
import math

def performActionMinMax(state: Tablerowumpus, player: int, depth: int):
    if player == 1:  # Agente
        moves = state.getAvailableMovesForMax(*state.agent_pos)
        best_move = None
        best_value = float('-inf')

        for move in moves:
            new_state = state.copy()
            new_state.move(move)  # Aplica el movimiento del agente

            # Llamamos a miniMax para evaluar el estado resultante después de que el agente se mueva
            _, value, _ = miniMax(new_state, 1, depth - 1, 2, -math.inf, math.inf)  # Cambia el jugador a MIN
            
            # Evaluar si encontramos un mejor movimiento
            if value > best_value:
                best_value = value
                best_move = move
        
        if best_move:  # Si encontramos un movimiento óptimo
            # Realiza el movimiento y verifica si el juego ha terminado
            game_over = state.move(best_move)
            if game_over:
                return state, best_move, state.utility(), True  # Si el juego terminó

            return state, best_move, state.utility(), state.isGameOver()  # Retorno del estado con la utilidad y estado del juego
    
    else:  # Tablero (hoyos)
        state.move_random_hole()  # Mueve un hoyo aleatoriamente
        game_over = state.isGameOver()  # Verifica si el juego ha terminado
        return state, "Hoyo movido", state.utility(), game_over
    
    return state, None, state.utility(), state.isGameOver()  # Retorno por defecto



In [72]:
def AIAction(state: Tablerowumpus, player: int):
    current_state = state.copy()
    
    if player == 1:  # Turno del agente
        moves = current_state.getAvailableMovesForMax(*current_state.agent_pos)
        
        if moves:
            best_move = None
            best_value = float('-inf')

            for move in moves:
                move_value = evaluate_move(current_state, move)
                
                # Si el movimiento lleva al agente a una celda con oro (incluso si hay hedor), priorizarlo.
                if current_state.matrix[current_state.agent_pos[0]][current_state.agent_pos[1]] == [4, 8, 9, 10]:
                    move_value += 100
                
                if move_value > best_value:
                    best_value = move_value
                    best_move = move
            
            
            game_over = current_state.move(best_move)
            return current_state, best_move, current_state.utility(), game_over
        
        else:
            return current_state, None, current_state.utility(), True
    
    else:  # Turno del tablero (mover un hoyo)
        current_state.move_random_hole()
        game_over = current_state.isGameOver()
        return current_state, "Hoyo movido", current_state.utility(), game_over

def evaluate_move(state: Tablerowumpus, move: str) -> float:
    temp_state = state.copy()
    temp_state.move(move)
    
    # Calcular utilidad basada en peligros y proximidad al oro
    utility_value = temp_state.utility()
    
    # Posición del oro
    gold_pos = next(((i, j) for i in range(temp_state.rows) for j in range(temp_state.cols) 
                     if temp_state.matrix[i][j] in [4, 8, 9, 10]), None)

    if gold_pos is None:
        # Si no hay oro, retornar solo la utilidad
        return utility_value

    # Distancia al oro
    agent_row, agent_col = temp_state.agent_pos
    distance_to_gold = abs(agent_row - gold_pos[0]) + abs(agent_col - gold_pos[1])
    
    # Bonificación significativa si se mueve hacia el oro
    if temp_state.matrix[temp_state.agent_pos[0]][temp_state.agent_pos[1]] in [4, 8, 9, 10]:  # Si está en la celda del oro
        return utility_value + 100  # Gran bonificación por encontrar el oro
    
    # Penalización ajustada por distancia al oro
    return utility_value - (distance_to_gold * 5)



def check_victory(state: Tablerowumpus) -> bool:
    return state.has_gold(state.agent_pos)  # Verificar si hay oro en la posición actual del agente

from IPython.display import display, clear_output, HTML

def run_wumpus_game_test():
    game = Tablerowumpus()
    game.setup_board()
    
    turn = 0
    last_move = "Inicio del juego"
    last_value = game.utility()
    
    game_over = False

    while not game_over and turn < 40: #evitar bucles infinitos (aunque no deberia suceder)        
        clear_output(wait=True)
        display(HTML(game.get_html()))
        print(f"Turno: {turn}")
        print(f"Último movimiento: {last_move}")
        print("\nPresiona Enter para continuar o 'q' para salir.")
        
        user_input = input()
        if user_input.lower() == 'q':
            print("Juego terminado por el usuario.")
            return
        
        turn += 1
        player = 1 if turn % 2 != 0 else 2
        
        new_state, move, value, game_over = AIAction(game, player)
        
        last_move = f"{'Agente' if player == 1 else 'Tablero'}: {move}"
        last_value = value
        game = new_state

    clear_output(wait=True)
    display(HTML(game.get_html()))
    if game_over:
        print("El juego ha terminado.")
    else:
        print("El juego ha terminado sin encontrar el oro.")

def print_board(state: Tablerowumpus):
    for row in state.getMatrix():
        print(' '.join(f"{cell:2d}" for cell in row))
    print(f"Posición del agente: {state.agent_pos}")
    print()

# Ejecutar la prueba
if __name__ == "__main__":
    run_wumpus_game_test()

,,,,,
,,,,,
,,,,,
,,,,,
,,,,,
,,,,,


El juego ha terminado.
